# Day 09. Exercise 02
# Metrics

## 0. Imports

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score, precision_score, recall_score, roc_auc_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import make_pipeline
import joblib
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from itertools import product
from tqdm import tqdm

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.concat([pd.read_csv('../data/day-of-week-not-scaled.csv'), pd.read_csv('../data/dayofweek.csv')['dayofweek']], axis=1)
X = df.drop(columns='dayofweek')
y = df['dayofweek']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [ ]:
svc = SVC(random_state=21, kernel='rbf', C=10, gamma='auto', class_weight='balanced', probability=True)

svc.fit(X_train, y_train)

y_pred = svc.predict(X_test)
y_prob = svc.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovo', average='weighted')

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.88757
precision is 0.88826
recall is 0.88757
roc_auc is 0.97825


## 3. Decision tree

1. The same task for decision tree

In [6]:
dt_model = DecisionTreeClassifier(random_state=21, criterion='gini', max_depth=21, class_weight='balanced')

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_prob = dt_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovo', average='weighted')

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.88462
precision is 0.88765
recall is 0.88462
roc_auc is 0.93528


## 4. Random forest

1. The same task for random forest.

In [7]:
rf_model = RandomForestClassifier(random_state=21, criterion='entropy', max_depth=24, class_weight='balanced')

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovo', average='weighted')

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.92604
precision is 0.92754
recall is 0.92604
roc_auc is 0.98939


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [8]:
rf_model = RandomForestClassifier(random_state=21, criterion='entropy', max_depth=24, class_weight='balanced')

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

test_analysis = pd.DataFrame({
    'y_true': y_test,
    'predictions': y_pred
})

test_analysis['is_error'] = (test_analysis['y_true'] != test_analysis['predictions']).astype(int)

days_map = {
    0: '0. Понедельник',
    1: '1.     Вторник',
    2: '2.       Среда',
    3: '3.     Четверг',
    4: '4.     Пятница',
    5: '5.     Суббота',
    6: '6. Воскресенье'
}
test_analysis['day_name'] = test_analysis['y_true'].map(days_map)

error_by_day = test_analysis.groupby('day_name').agg(
    test_samples=('is_error', 'count'),  # Сколько тестовых объектов было в этот день
    total_errors=('is_error', 'sum'),     # Сколько раз модель ошиблаcь
    error_rate=('is_error', 'mean')       # Доля ошибок (Error Rate)
).reset_index()

error_by_day = error_by_day.sort_values(by='error_rate', ascending=False)

error_by_day['error_rate_%'] = (error_by_day['error_rate'] * 100).round(2)
error_by_day['accuracy_%'] = ((1 - error_by_day['error_rate']) * 100).round(2)

print(error_by_day[['day_name', 'test_samples', 'total_errors', 'error_rate_%', 'accuracy_%']].to_string(index=False))

      day_name  test_samples  total_errors  error_rate_%  accuracy_%
0. Понедельник            27             6         22.22       77.78
4.     Пятница            21             3         14.29       85.71
5.     Суббота            54             5          9.26       90.74
1.     Вторник            55             4          7.27       92.73
2.       Среда            30             2          6.67       93.33
3.     Четверг            80             3          3.75       96.25
6. Воскресенье            71             2          2.82       97.18


In [11]:
lab_cols = [c for c in X_test.columns if c.startswith('labname')]
test_analysis['labname'] = X_test[lab_cols].idxmax(axis=1).str.replace('labname_', '')

user_cols = [c for c in X_test.columns if c.startswith('uid')]
test_analysis['user'] = X_test[user_cols].idxmax(axis=1).str.replace('uid_', '')

In [13]:
error_by_labname = test_analysis.groupby('labname').agg(
    test_samples=('is_error', 'count'),  # Сколько тестовых объектов было в этот день
    total_errors=('is_error', 'sum'),     # Сколько раз модель ошиблаcь
    error_rate=('is_error', 'mean')       # Доля ошибок (Error Rate)
).reset_index()

error_by_labname = error_by_labname.sort_values(by='error_rate', ascending=False)

error_by_labname['error_rate_%'] = (error_by_labname['error_rate'] * 100).round(2)
error_by_labname['accuracy_%'] = ((1 - error_by_labname['error_rate']) * 100).round(2)

print(error_by_labname[['labname', 'test_samples', 'total_errors', 'error_rate_%', 'accuracy_%']].to_string(index=False))

 labname  test_samples  total_errors  error_rate_%  accuracy_%
   lab03             1             1        100.00        0.00
  laba04            35             6         17.14       82.86
  lab05s             6             1         16.67       83.33
 laba06s            15             2         13.33       86.67
  laba06             9             1         11.11       88.89
 laba04s            25             2          8.00       92.00
code_rvw            13             1          7.69       92.31
project1           186            10          5.38       94.62
  laba05            47             1          2.13       97.87
  lab03s             1             0          0.00      100.00


In [15]:
error_by_user = test_analysis.groupby('user').agg(
    test_samples=('is_error', 'count'),  # Сколько тестовых объектов было в этот день
    total_errors=('is_error', 'sum'),     # Сколько раз модель ошиблаcь
    error_rate=('is_error', 'mean')       # Доля ошибок (Error Rate)
).reset_index()

error_by_user = error_by_user.sort_values(by='error_rate', ascending=False)

error_by_user['error_rate_%'] = (error_by_user['error_rate'] * 100).round(2)
error_by_user['accuracy_%'] = ((1 - error_by_user['error_rate']) * 100).round(2)

print(error_by_user[['user', 'test_samples', 'total_errors', 'error_rate_%', 'accuracy_%']].to_string(index=False))

   user  test_samples  total_errors  error_rate_%  accuracy_%
user_22             1             1        100.00        0.00
 user_6             4             2         50.00       50.00
user_16             5             1         20.00       80.00
user_27             6             1         16.67       83.33
user_18             6             1         16.67       83.33
 user_3            14             2         14.29       85.71
user_30             8             1         12.50       87.50
user_31            18             2         11.11       88.89
 user_2            28             3         10.71       89.29
user_19            19             2         10.53       89.47
user_29            11             1          9.09       90.91
user_24            11             1          9.09       90.91
user_25            22             2          9.09       90.91
user_10            12             1          8.33       91.67
 user_4            27             2          7.41       92.59
user_13 

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [17]:
def evaluate_models(
        models, list_of_params, X_train, y_train, X_test, y_test
):
    metrics = {}
    for model, params in zip(models, list_of_params):
        model = model(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovo', average='weighted')
        metrics[model.__class__.__name__] = {
            'accuracy': round(accuracy, 5),
            'precision' : round(precision,5),
            'recall' : round(recall,5),
            'roc_auc' : round(roc_auc, 5)
        }
    return metrics

In [18]:
models = [SVC, DecisionTreeClassifier, RandomForestClassifier]


list_of_params = [
    {'C': 10, 'kernel': 'rbf', 'probability': True, 'gamma':'auto', 'class_weight':'balanced', 'random_state': 21},
     {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21},
    {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}
]

res = evaluate_models(models, list_of_params, X_train, y_train, X_test, y_test)
print(res)

{'SVC': {'accuracy': 0.88757, 'precision': 0.88826, 'recall': 0.88757, 'roc_auc': 0.97825}, 'DecisionTreeClassifier': {'accuracy': 0.89053, 'precision': 0.89416, 'recall': 0.89053, 'roc_auc': 0.93818}, 'RandomForestClassifier': {'accuracy': 0.92604, 'precision': 0.92772, 'recall': 0.92604, 'roc_auc': 0.98955}}
